In [ ]:
from pathlib import Path

from ocularrigidity.data.io import load_cube
from ocularrigidity.registration.fused import segment_and_register
from ocularrigidity.segmentation.utils import (
    get_choroid_segmentation_model,
    get_registration_model,
)
from ocularrigidity.registration.config import RegistrationConfig


In [ ]:
MAX_FRAMES = 200
DEVICE = "cuda"  # or "cpu"
segmentation_model = get_choroid_segmentation_model().to(DEVICE)
registration_model = get_registration_model().to(DEVICE)

# Path to a video file
path = Path(
    "/mnt/smb/CZMI1166658438/2021-01-20/Deformation/OD/"
)  # <- must point to a folder containing a cube.bin and a timestamp.txt
frames = load_cube(path)

# Usually we drop some of the frames
frames = frames[
    20:MAX_FRAMES
]  # Skipping the first 20 frames limiting the computation to MAX_FRAMES (-20) frames


In [ ]:
config = RegistrationConfig(
    fused_batch_size=8, batch_size=16, keep_largest_cc=False
)  # Fused batch size is the number of frames to process at once for segmentation and registration, while batch size is the number of frames to process at once for warping.

registered_and_segmented_results = segment_and_register(
    frames,
    segmentation_model,
    registration_model,
    config=config,
    device=DEVICE,
    verbose=True,
)


In [ ]:
from ocularrigidity.data.compression import cube_to_mkv_lossless, cube_to_mp4


registered_and_segmented_results.raw_masks  # Segmentation of the choroid for each frame
registered_and_segmented_results.registered_masks  # Registered segmentation of the choroid for each frame
registered_and_segmented_results.registered_frames  # Registered frames
cube_to_mp4(
    registered_and_segmented_results.registered_frames,
    "result.mp4",
    crf=28,
    fps=60,
    verbose=True,
)  # Lossy, lower crf means better quality but bigger file size. 28 is a good compromise for visualisation.

cube_to_mkv_lossless(
    registered_and_segmented_results.registered_frames, "result.mkv", verbose=True
)  # Lossless, bigger file size but no loss of information. Good for further processing.
